# Channel Revenue Summary — SQL

SQL-cell equivalent of `channel_revenue_summary.ipynb`, using `%%sql`
cells instead of the DataFrame API. Same result: reads
`commerce.store_sales`, `commerce.catalog_sales`, and
`commerce.web_sales`, joins in `commerce.date_dim` and
`commerce.promotion`, and persists
`commerce.channel_revenue_summary_v1`.

Requires the raw tables to already exist — run
`data-generators/tpcds_retail_ingest.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: revenue and margin by channel and year

In [ ]:
%%sql
DROP TABLE IF EXISTS commerce.channel_revenue_summary_v1;

CREATE TABLE commerce.channel_revenue_summary_v1 AS
WITH channel_sales AS (
    SELECT
        ss_sold_date_sk AS sold_date_sk,
        ss_promo_sk AS promo_sk,
        ss_ext_sales_price AS ext_sales_price,
        ss_net_profit AS net_profit,
        'store' AS channel
    FROM commerce.store_sales
    UNION ALL
    SELECT
        cs_sold_date_sk,
        cs_promo_sk,
        cs_ext_sales_price,
        cs_net_profit,
        'catalog'
    FROM commerce.catalog_sales
    UNION ALL
    SELECT
        ws_sold_date_sk,
        ws_promo_sk,
        ws_ext_sales_price,
        ws_net_profit,
        'web'
    FROM commerce.web_sales
),
joined AS (
    SELECT
        cs.channel,
        d.d_year,
        cs.ext_sales_price,
        cs.net_profit,
        CASE WHEN p.p_promo_sk IS NOT NULL THEN cs.ext_sales_price ELSE 0.0 END AS promo_revenue
    FROM channel_sales cs
    LEFT JOIN commerce.date_dim d ON cs.sold_date_sk = d.d_date_sk
    LEFT JOIN commerce.promotion p ON cs.promo_sk = p.p_promo_sk
)
SELECT
    channel,
    d_year,
    COUNT(*) AS line_item_count,
    ROUND(SUM(ext_sales_price), 2) AS total_revenue,
    ROUND(SUM(net_profit), 2) AS total_profit,
    ROUND(SUM(net_profit) * 100.0 / SUM(ext_sales_price), 1) AS profit_margin_pct,
    ROUND(SUM(promo_revenue) * 100.0 / SUM(ext_sales_price), 1) AS promo_revenue_share_pct
FROM joined
GROUP BY channel, d_year
ORDER BY d_year, channel

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_rows FROM commerce.channel_revenue_summary_v1

In [ ]:
%%sql
SELECT * FROM commerce.channel_revenue_summary_v1
ORDER BY d_year, channel
LIMIT 30